# 허깅페이스 트랜스포머 모델 환경 구축

In [1]:
import sys
print(sys.version)

import transformers
import datasets
import pandas

print("transformers:",transformers.__version__)
print("datasets:",datasets.__version__)
print("pandas:",pandas.__version__)
print("환경설정 완료")

3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]
transformers: 5.4.0
datasets: 4.8.4
pandas: 2.3.3
환경설정 완료


# 감정분석 모델 (영어)

In [2]:
from transformers import pipeline

pipe = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=0
)

result = pipe("I really enjoyed this class.")
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998738765716553}]


# 추론 서비스

In [8]:
import os
import requests
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("HF_TOKEN")

In [9]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

result = client.translation(
    "나는 볼프강이고, 베를린에 살고 있습니다",
    model="Helsinki-NLP/opus-mt-ko-en",
)

In [9]:
print(result)

TranslationOutput(translation_text="I'm Wolfgang, and I live in Berlin.")


# text-classification

In [14]:
from transformers import pipeline

# task만 지정하면 기본 모델 자동 로드
# 기본 모델: distilbert/distilbert-base-uncased-finetuned-sst-2-english
classifier = pipeline("text-classification")

result = classifier("No model was supplied, defaulted to distilbert")
print(result)
# [{'label': 'POSITIVE', 'score': 0.9998}]

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9982872605323792}]


In [21]:
domain_text = [
  "I'm hungry but only thing I've got is Dubai Zzondeuk cookie.",
  "If I eat this Dubai Zzondeuk cookie, I might get fat.",
  "Whatever, I don't care."
]

In [22]:
fin_classifier = pipeline("text-classification", model="ProsusAI/finbert")
result_fin = fin_classifier(domain_text)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
print(result_fin)

[{'label': 'neutral', 'score': 0.8997340202331543}, {'label': 'neutral', 'score': 0.5278830528259277}, {'label': 'neutral', 'score': 0.8281916379928589}]


In [24]:
for text, result in zip(domain_text, result_fin):
  print(f"입력: {text}")
  print(f"결과: {result['label']} ({result['score']:4f})")

입력: I'm hungry but only thing I've got is Dubai Zzondeuk cookie.
결과: neutral (0.899734)
입력: If I eat this Dubai Zzondeuk cookie, I might get fat.
결과: neutral (0.527883)
입력: Whatever, I don't care.
결과: neutral (0.828192)


# 금융 텍스트 분석

In [ ]:
kr_classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")
kr_result = kr_classifier("엄마가 해준 밥을 안 먹으면 배고파요")
print(kr_result)

config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

c:\Users\Admin\miniconda3\envs\hf-nlp\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--snunlp--KR-FinBert-SC. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: snunlp/KR-FinBert-SC
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors:   0%|          | 0.00/406M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': 'neutral', 'score': 0.9994363188743591}]


In [26]:
kr_classifier = pipeline("sentiment-analysis", model="tabularisai/multilingual-sentiment-analysis")
kr_result = kr_classifier("엄마가 해준 밥을 안 먹으면 배고파요")
print(kr_result)

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

c:\Users\Admin\miniconda3\envs\hf-nlp\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--tabularisai--multilingual-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[{'label': 'Negative', 'score': 0.5031715035438538}]


In [27]:
kr_classifier = pipeline("sentiment-analysis", model="tabularisai/multilingual-sentiment-analysis")
kr_result = kr_classifier("두바이 쫀득 봄동 버터떡 나랑 먹으러 갈래?")
print(kr_result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'Neutral', 'score': 0.6036347150802612}]


# fill-mask

In [7]:
fill_mask = pipeline("fill-mask", model="snunlp/KR-FinBert")
fill_mask("배고플 땐 [MASK]를 먹어야지")


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

c:\Users\Admin\miniconda3\envs\hf-nlp\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--snunlp--KR-FinBert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: snunlp/KR-FinBert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/406M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'score': 0.22066307067871094,
  'token': 18433,
  'token_str': '고기',
  'sequence': '배고플 땐 고기 를 먹어야지'},
 {'score': 0.08998339623212814,
  'token': 13329,
  'token_str': '커피',
  'sequence': '배고플 땐 커피 를 먹어야지'},
 {'score': 0.03245833143591881,
  'token': 2976,
  'token_str': '배',
  'sequence': '배고플 땐 배 를 먹어야지'},
 {'score': 0.028142616152763367,
  'token': 13991,
  'token_str': '김치',
  'sequence': '배고플 땐 김치 를 먹어야지'},
 {'score': 0.018684331327676773,
  'token': 4484,
  'token_str': '피',
  'sequence': '배고플 땐 피 를 먹어야지'}]

# 질의응답

In [ ]:
# 질문과 지문 정의
qa_samples = [
    {
        "question": "What is the capital of France?",
        "context": "France is a country in Western Europe. Its capital is Paris."
    },
    {
        "question": "Who invented the telephone?",
        "context": "The telephone was invented by Alexander Graham Bell in 1876."
    },
    {
        "question": "What language does Python use for indentation?",
        "context": "Python uses whitespace indentation to define code blocks."
    },
]

In [18]:
# 질문과 지문 정의
qa_samples = [
    {
        "question": "What does the twinkling streetlamp whisper at midnight?",
        "context": "The streetlamp glows softly and whispers that the night is full of forgotten stories."
    },
    {
        "question": "Which flavor does the invisible bubblegum taste like?",
        "context": "Invisible bubblegum tastes like laughter and summer rain, with a hint of nostalgia."
    },
    {
        "question": "Why does the sleepy moon borrow a blanket from the clouds?",
        "context": "The moon gets chilly on clear nights, so it borrows a cloud blanket to keep its glow cozy."
    }
]


In [17]:
import torch 


for sample in qa_samples:
    inputs = tokenizer(sample["question"], sample["context"], return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    # 정답 시작/끝 위치 추출
    start = torch.argmax(outputs.start_logits)
    end   = torch.argmax(outputs.end_logits) + 1
    
    # logits → softmax → 확률값
    start_prob = torch.softmax(outputs.start_logits, dim=-1)
    end_prob   = torch.softmax(outputs.end_logits,   dim=-1)

    # 각 최댓값이 해당 위치의 확률 = score
    start_score = torch.max(start_prob).item()
    end_score   = torch.max(end_prob).item()

    # 최종 score = start * end 확률의 곱
    score = start_score * end_score
    
    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    )

    print(f"Q: {sample['question']}")
    print(f"A: {answer}")
    print(f"Score  : {score:.4f}")
    print()

Q: Why did the chicken cross the road?
A: The chicken wanted to get to the other side
Score  : 0.4546

Q: What color is the sky on a clear day?
A: blue
Score  : 0.9875

Q: What does a clock do?
A: shows the time with hands or numbers
Score  : 0.8185



# 번역 모델

In [ ]:
# pip install sentencepiece --break-system-packages

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 16.9 MB/s  0:00:00


In [3]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

c:\Users\Admin\miniconda3\envs\hf-nlp\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-ko-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

In [4]:
sentences = [
    "안녕하세요, 저는 인공지능을 공부하고 있습니다.",
    "오늘 날씨가 정말 좋네요.",
    "한국의 경제 성장률이 올해 2%를 기록했습니다.",
]

# 배치 번역
inputs = tokenizer(sentences, return_tensors="pt", padding=True)
translated = model.generate(**inputs)
results = tokenizer.batch_decode(translated, skip_special_tokens=True)

for src, tgt in zip(sentences, results):
    print(f"입력: {src}")
    print(f"번역: {tgt}")
    print()

입력: 안녕하세요, 저는 인공지능을 공부하고 있습니다.
번역: Hi, I'm studying AI.

입력: 오늘 날씨가 정말 좋네요.
번역: It's a great day.

입력: 한국의 경제 성장률이 올해 2%를 기록했습니다.
번역: South Korea's economic growth rate is 2% this year.



In [5]:
sentences = [
    "고기 주세요.",
    "오늘도 운동을 안했다간 근손실이 나고 말 것입니다.",
    "프로틴은 당신의 삶에 근육을 추가해 줍니다.",
]

# 배치 번역
inputs = tokenizer(sentences, return_tensors="pt", padding=True)
translated = model.generate(**inputs)
results = tokenizer.batch_decode(translated, skip_special_tokens=True)

for src, tgt in zip(sentences, results):
    print(f"입력: {src}")
    print(f"번역: {tgt}")
    print()

입력: 고기 주세요.
번역: Let's get some meat.

입력: 오늘도 운동을 안했다간 근손실이 나고 말 것입니다.
번역: If you haven't exercised today, you're going to have loss of muscle.

입력: 프로틴은 당신의 삶에 근육을 추가해 줍니다.
번역: Protin adds muscle to your life.



In [10]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

text = "Artificial intelligence is changing how we learn and work."

result = client.translation(
    text,
    model="Helsinki-NLP/opus-mt-tc-big-en-ko",
)

print("원문:", text)
print("번역:", result.translation_text)


원문: Artificial intelligence is changing how we learn and work.
번역: ll popular 즐거운 universalpatriot 끝 세그먼트입니다.


# 영어 요약 모델

In [14]:
from transformers import BartForConditionalGeneration, BartTokenizer
import torch

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

article = """
Good morning! And in case I don't see ya, good afternoon, good evening, and good night! Did you see the news? They say a satellite fell from the sky, but I think it was just a very large bird with a metal problem. My lawn is green, the sun is shining, and my neighbors are waving exactly like they did yesterday, and the day before, and the day before that. It’s a beautiful day in Seahaven, isn't it? Even if the wind smells like hairspray and the horizon looks like a painted wall.
"""

inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True)

with torch.no_grad():
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100,
        num_beams=4,          # 빔 서치로 품질 향상
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"원문: {article.strip()}")
print(f"요약: {summary}")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


원문: Good morning! And in case I don't see ya, good afternoon, good evening, and good night! Did you see the news? They say a satellite fell from the sky, but I think it was just a very large bird with a metal problem. My lawn is green, the sun is shining, and my neighbors are waving exactly like they did yesterday, and the day before, and the day before that. It’s a beautiful day in Seahaven, isn't it? Even if the wind smells like hairspray and the horizon looks like a painted wall.
요약: "It’s a beautiful day in Seahaven, isn't it? Even if the wind smells like hairspray and the horizon looks like a painted wall. Did you see the news? They say a satellite fell from the sky, but I think it was just a very large bird with a metal problem"


In [15]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

result = client.summarization(
    "Good morning! And in case I don't see ya, good afternoon, good evening, and good night! Did you see the news? They say a satellite fell from the sky, but I think it was just a very large bird with a metal problem. My lawn is green, the sun is shining, and my neighbors are waving exactly like they did yesterday, and the day before, and the day before that. It’s a beautiful day in Seahaven, isn't it? Even if the wind smells like hairspray and the horizon looks like a painted wall.",
    model="facebook/bart-large-cnn",
)

In [16]:
print(result)

SummarizationOutput(summary_text='"It’s a beautiful day in Seahaven, isn\'t it? Even if the wind smells like hairspray and the horizon looks like a painted wall" "They say a satellite fell from the sky, but I think it was just a very large bird with a metal problem"')


In [18]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "google/pegasus-xsum"
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/pegasus-xsum",
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)

input_text = """Good morning! And in case I don't see ya, good afternoon, good evening, and good night! Did you see the news? They say a satellite fell from the sky, but I think it was just a very large bird with a metal problem. My lawn is green, the sun is shining, and my neighbors are waving exactly like they did yesterday, and the day before, and the day before that. It’s a beautiful day in Seahaven, isn't it? Even if the wind smells like hairspray and the horizon looks like a painted wall."""
input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)

output = model.generate(**input_ids, cache_implementation="static")
print(tokenizer.decode(output[0], skip_special_tokens=True))

config.json: 0.00B [00:00, ?B/s]

c:\Users\Admin\miniconda3\envs\hf-nlp\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--google--pegasus-xsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

It's a beautiful day in Seahaven, isn't it?


# 임베딩

In [25]:
pip install -U sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [26]:
from sentence_transformers import SentenceTransformer
from torch.nn.functional import cosine_similarity

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The weather is lovely today.",   # 날씨 문장 1
    "It's so sunny outside!",          # 날씨 문장 2
    "He drove to the stadium.",        # 관계없는 문장
]

# 문장 → 384차원 벡터로 변환
embeddings = model.encode(sentences, convert_to_tensor=True)
print(f"임베딩 shape: {embeddings.shape}")  # (3, 384)
print()

# 문장 간 코사인 유사도 계산
pairs = [(0, 1), (0, 2), (1, 2)]
for i, j in pairs:
    score = cosine_similarity(
        embeddings[i].unsqueeze(0),
        embeddings[j].unsqueeze(0)
    ).item()
    print(f"[{sentences[i]}]")
    print(f"[{sentences[j]}]")
    print(f"유사도: {score:.4f}")
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Admin\miniconda3\envs\hf-nlp\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 shape: torch.Size([3, 384])

[The weather is lovely today.]
[It's so sunny outside!]
유사도: 0.6660

[The weather is lovely today.]
[He drove to the stadium.]
유사도: 0.1046

[It's so sunny outside!]
[He drove to the stadium.]
유사도: 0.1411



# 텍스트 생성 (할루시네이션이 아쉬움)

In [37]:
def generate(prompt, do_sample=False, num_beams=1, temperature=1.0):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=do_sample,
            num_beams=num_beams,
            temperature=0.1,
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"사용 디바이스: {device}")



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

사용 디바이스: cuda


In [40]:
prompt = "어깨가 넓어보이고 싶을 때 가장 중요한 근육은 무엇일까?"

print(f"Q: {prompt}\n")
print(f"[탐욕 탐색]\n{generate(prompt)}\n")
print(f"[빔 서치]\n{generate(prompt, num_beams=4)}\n")
print(f"[샘플링]\n{generate(prompt, do_sample=True, temperature=0.3)}\n")

Q: 어깨가 넓어보이고 싶을 때 가장 중요한 근육은 무엇일까?

[탐욕 탐색]
어깨가 넓어 보이는 이유는 여러 가지 요인에 의해 발생합니다. 이 중 하나로는 몸의 흐름과 운동이 있습니다. 어깨가 넓어지는 것은 몸이 흐르는 방향을 정확하게 유지하고, 이를 위해 몸이 움직여야 합니다.

또한, 어깨가 넓어지는 것은 몸의 근육이 활성화되고, 이로 인해 몸이 더 강력하고 힘을 발휘할 수 있게 됩니다. 또한, 어깨가 넓어지는 것은 몸의 신체 구조를 조절하는 데에도 중요하며, 이는 몸의 건강과 체중 관리에 큰 도움이 될 것입니다.

따라서, 어깨가 넓어지는 것이 중요하다고 생각하면, 몸의 근육이 활성화되어, 몸이 더 강력하고 힘을 발휘할 수 있도록 하는 것이 중요합니다. 이러한 근육은 몸의 흐름과 운동을 지원하고, 건강과 체중 관리를 돕습니다.

[빔 서치]
어깨가 넓어보이고 싶을 때 가장 중요한 근육은 허리 근육입니다. 허리 근육은 허리와 어깨 사이에 위치하며, 이 근육의 활동이 어깨가 넓어보이는 원인 중 하나입니다. 허리 근육의 활동은 다음과 같은 효과를 가져옵니다:

1. 어깨가 넓어보이기: 허리 근육의 활동은 어깨가 넓어질 수 있습니다. 이는 허리 근육이 허리와 어깨 사이에 위치해 있으며, 이 근육의 활동이 이로 인해 허리와 어깨가 더 넓어질 수 있게 해줍니다.

2. 허리가 편안해지기: 허리 근육의 활동은 허리가 편안해질 수 있습니다. 이는 허리 근육이 허리와 어깨 사이에 위치해 있으며, 이 근육의 활동이 이로 인해 허리와 어깨가 더 편안해질 수 있게 해줍니다.

3. 허리 근육의 편안함: 허리 근육의 활동은 허리 근육의 편안함을 높일 수 있습니다. 이는 허리 근육이 허리와 어깨 사이에 위치해 있으며, 이 근육의 활동이 이로 인해 허리 근육의 편안함을 높일 수 있게 해줍니다.

4. 허리 근육의 편안함: 허리 근

[샘플링]
어깨가 넓어 보이는 이유는 여러 가지 요인에 의해 발생합니다. 이 중 하나로는 몸의 흐름과 운동이 있습니다. 어깨가 넓어지는 것은 몸이 흐